# Sesión 5 · LLMs: generación, cuantización y LoRA

Usa y adapta un LLM pequeño con los recursos de Colab (GPU T4).

Objetivos:
1. Controlar la generación.
2. Cuantizar en 4 bits y aplicar LoRA.
3. Hacer SFT y medir el antes/después.

In [ ]:
%pip install -q transformers datasets accelerate peft bitsandbytes trl

## 1 · Generación controlada

In [ ]:
from transformers import pipeline

gen = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M")
print(gen("Escribe una frase de bienvenida para una tienda:",
          max_new_tokens=30, temperature=0.7, top_p=0.9)[0]["generated_text"])

## 2 · Cargar cuantizado en 4 bits

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

nombre = "HuggingFaceTB/SmolLM2-360M"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)

modelo = AutoModelForCausalLM.from_pretrained(nombre, quantization_config=bnb, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(nombre)
tokenizer.pad_token = tokenizer.eos_token
print("modelo cargado en 4 bits")

## 3 · Aplicar LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

modelo = prepare_model_for_kbit_training(modelo)
lora = LoraConfig(r=8, lora_alpha=16, target_modules="all-linear",
                  lora_dropout=0.05, task_type="CAUSAL_LM")
modelo = get_peft_model(modelo, lora)
modelo.print_trainable_parameters()  # << 1 % de parámetros entrenables

## 4 · SFT con trl (dataset de instrucciones)

Prepara un dataset pequeño en formato chat (rol user/assistant) y entrena con `SFTTrainer`.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

ejemplos = [
    {"mensajes": [
        {"role": "user", "content": "Saluda al cliente"},
        {"role": "assistant", "content": "¡Hola! Bienvenido a SentimentLab. ¿En qué puedo ayudarte?"},
    ]},
    # TODO: añade más ejemplos con tu estilo/formato fijo
]
dataset = Dataset.from_list(ejemplos)

trainer = SFTTrainer(
    model=modelo,
    args=SFTConfig(output_dir="./out", max_seq_length=256, num_train_epochs=1),
    train_dataset=dataset,
)
trainer.train()

## Práctica y reto

1. Ajusta el modelo para que responda con un **estilo/formato fijo**.
2. **Mide antes/después** con 10 preguntas de prueba.

> Entrega: notebook ejecutado + comparativa.

In [ ]:
# TODO: genera antes y después del ajuste y compara